In [5]:
pip install merlin-vlm

Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install datasets huggingface_hub nibabel numpy torch torchvision

Note: you may need to restart the kernel to use updated packages.


In [7]:
import os

PROJECT_ROOT = os.path.expanduser("~/Documents/GitHub/fyp-3d-ct/models/merlin")
DATA_ROOT    = os.path.join(PROJECT_ROOT, "data")
VOLUMES_DIR  = os.path.join(DATA_ROOT, "data_volumes")
REXCT_DIR    = os.path.join(DATA_ROOT, "rexgrounding-ct")
DATASET_JSON = os.path.join(REXCT_DIR, "dataset.json")

# Find all .nii.gz files on disk
ct_index = {}
for root, dirs, files in os.walk(VOLUMES_DIR):
    for f in files:
        if f.endswith(".nii.gz"):
            ct_index[f] = os.path.join(root, f)

print(f"CT files found on disk: {len(ct_index)}")
print("\nFiles available:")
for name, path in ct_index.items():
    print(f"  {name}")

CT files found on disk: 169

Files available:
  train_108_a_2.nii.gz
  train_34_a_2.nii.gz
  train_33_a_1.nii.gz
  train_106_a_1.nii.gz
  train_93_c_2.nii.gz
  train_93_d_2.nii.gz
  train_67_a_2.nii.gz
  train_60_d_1.nii.gz
  train_56_a_1.nii.gz
  train_197_a_2.nii.gz
  train_563_a_2.nii.gz
  train_107_c_2.nii.gz
  train_107_b_2.nii.gz
  train_107_a_1.nii.gz
  train_332_a_1.nii.gz
  train_564_a_1.nii.gz
  train_368_a_1.nii.gz
  train_357_a_2.nii.gz
  train_68_a_2.nii.gz
  train_196_a_2.nii.gz
  train_506_a_1.nii.gz
  train_392_a_1.nii.gz
  train_366_a_2.nii.gz
  train_61_a_2.nii.gz
  train_59_a_2.nii.gz
  train_198_a_2.nii.gz
  train_66_a_2.nii.gz
  train_445_a_1.nii.gz
  train_214_a_2.nii.gz
  train_9_a_1.nii.gz
  train_473_a_2.nii.gz
  train_225_a_1.nii.gz
  train_240_a_2.nii.gz
  train_247_a_2.nii.gz
  train_6_a_2.nii.gz
  train_279_a_2.nii.gz
  train_246_b_1.nii.gz
  train_428_a_2.nii.gz
  train_277_a_1.nii.gz
  train_284_a_1.nii.gz
  train_252_b_2.nii.gz
  train_403_a_2.nii.gz
  t

In [8]:
import json

with open(DATASET_JSON) as f:
    metadata = json.load(f)

samples = metadata if isinstance(metadata, list) else list(metadata.values())[0]

# Find which JSON entries match your local files
matched = [s for s in samples if s["name"] in ct_index]
print(f"Matched {len(matched)} samples\n")
for s in matched:
    print(f"  name     : {s['name']}")
    print(f"  findings : {s['findings']}")
    print(f"  shape    : {s['shape']}")
    print(f"  protocol : {s['protocol']}")
    print()

Matched 169 samples

  name     : train_234_a_1.nii.gz
  findings : {'0': 'Minimal peribronchial thickening in both lungs', '1': 'Mosaic attenuation pattern in both lungs', '2': 'Minimal interlobular septal and interstitial thickening in the right middle lobe', '3': 'Honeycomb pattern in the right middle lobe', '4': 'Subcentimeter nodules in both lungs'}
  shape    : [512, 512, 303]
  protocol : protocol1

  name     : train_115_a_2.nii.gz
  findings : {'0': 'Emphysematous changes in both lungs, more pronounced in the left upper lobe', '1': 'Subcentimeter nodules in both lungs'}
  shape    : [512, 512, 487]
  protocol : protocol1

  name     : train_183_a_2.nii.gz
  findings : {'0': 'Mild bronchiectasis with peribronchial thickening in the lower lobes of both lungs', '1': 'Patchy ground glass opacities in the paracardiac region of the right middle lobe'}
  shape    : [512, 512, 269]
  protocol : protocol1

  name     : train_272_a_2.nii.gz
  findings : {'0': 'Scattered patches of groun

In [9]:
import nibabel as nib
import numpy as np
import torch

def load_and_preprocess_ct(nii_path):
    img    = nib.load(nii_path)
    volume = img.get_fdata()
    volume = np.clip(volume, -1000, 400)
    volume = (volume + 1000) / 1400.0
    volume = np.nan_to_num(volume, nan=0.0)
    tensor = torch.tensor(volume, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
    return tensor, img.affine, volume   # also return affine + raw for mask saving

In [10]:
from merlin import Merlin

model = Merlin(ImageEmbedding=True)
model.eval()
print("Merlin loaded!")

/Users/pasanthilakshana/Documents/GitHub/fyp-3d-ct/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:
import os
import json
import numpy as np
import nibabel as nib
import torch

os.makedirs("results", exist_ok=True)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def run_full_pipeline(sample: dict, model, ct_index: dict, out_dir="results"):
    name     = sample["name"]
    findings = sample["findings"]           # {"0": "text...", "1": "text..."}
    shape    = sample["shape"]
    cats     = sample.get("categories", {})
    pixels   = sample.get("pixels", {})
    protocol = sample.get("protocol", "unknown")

    ct_path = ct_index[name]
    print(f"\n{'='*55}")
    print(f"Processing : {name}")
    print(f"Shape      : {shape}")
    print(f"Protocol   : {protocol}")
    print(f"Findings   : {len(findings)}")
    print(f"{'='*55}")

    # ── Load CT ──────────────────────────────────────────────
    ct_tensor, affine, volume_norm = load_and_preprocess_ct(ct_path)
    print(f"  CT tensor shape : {ct_tensor.shape}")

    # ── Get embedding from Merlin ─────────────────────────────
    with torch.no_grad():
        embedding = model(ct_tensor)

    emb_np = embedding.cpu().numpy()
    print(f"  Embedding shape : {emb_np.shape}")

    # ── Build localization mask per finding ───────────────────
    # Merlin gives a global embedding — we create a simple
    # intensity-based localization mask as a proxy heatmap.
    # Each slice's mean activation becomes a weight.
    H, W, D = volume_norm.shape
    activation_volume = np.zeros((H, W, D), dtype=np.float32)

    emb_1d = emb_np.flatten()
    emb_scalar = float(np.mean(np.abs(emb_1d)))  # scalar 0-1 range

    for d in range(D):
        slc = volume_norm[:, :, d]
        # High HU regions (soft tissue, nodules) get higher activation
        activation_volume[:, :, d] = slc * emb_scalar

    # Threshold at top 5% to get binary mask
    threshold = np.percentile(activation_volume, 95)
    binary_mask = (activation_volume >= threshold).astype(np.uint8)
    print(f"  Mask voxels active : {binary_mask.sum()}")

    # ── Build report ──────────────────────────────────────────
    finding_lines = []
    for idx, text in findings.items():
        cat     = cats.get(idx, "unknown")
        px      = pixels.get(idx, "N/A")
        finding_lines.append(
            f"  Finding {int(idx)+1} [Category {cat}]:\n"
            f"    {text}\n"
            f"    Estimated pixel count: {px}"
        )

    report = f"""
MERLIN CT ANALYSIS REPORT
{'='*55}
File     : {name}
Protocol : {protocol}
Shape    : {shape[0]} x {shape[1]} x {shape[2]} voxels

EMBEDDING
  Shape  : {emb_np.shape}
  Norm   : {float(np.linalg.norm(emb_np)):.4f}
  Mean   : {float(np.mean(emb_np)):.4f}
  Std    : {float(np.std(emb_np)):.4f}

FINDINGS ({len(findings)} total)
{chr(10).join(finding_lines)}

LOCALIZATION MASK
  Active voxels : {binary_mask.sum()}
  Threshold     : {threshold:.4f} (top 5% activation)
  Coverage      : {100*binary_mask.sum()/(H*W*D):.2f}% of volume
{'='*55}
"""
    print(report)

    # ── Save all 4 outputs ────────────────────────────────────
    stem = name.replace(".nii.gz", "")
    out  = os.path.join(out_dir, stem)
    os.makedirs(out, exist_ok=True)

    # 1. Report (.txt)
    report_path = os.path.join(out, "report.txt")
    with open(report_path, "w") as f:
        f.write(report)
    print(f"  ✓ Report saved    → {report_path}")

    # 2. Embeddings (.npz)
    emb_path = os.path.join(out, "embedding.npz")
    np.savez(emb_path, embedding=emb_np, finding_texts=list(findings.values()))
    print(f"  ✓ Embedding saved → {emb_path}")

    # 3. Localization mask (.nii.gz)
    mask_nii  = nib.Nifti1Image(binary_mask, affine)
    mask_path = os.path.join(out, "localization_mask.nii.gz")
    nib.save(mask_nii, mask_path)
    print(f"  ✓ Mask saved      → {mask_path}")

    # 4. Findings (.json)
    findings_data = {
        "name"      : name,
        "protocol"  : protocol,
        "shape"     : shape,
        "findings"  : findings,
        "categories": cats,
        "pixels"    : pixels,
        "embedding_norm": float(np.linalg.norm(emb_np)),
        "mask_active_voxels": int(binary_mask.sum())
    }
    findings_path = os.path.join(out, "findings.json")
    with open(findings_path, "w") as f:
        json.dump(findings_data, f, indent=2)
    print(f"  ✓ Findings saved  → {findings_path}")

    return {
        "name"     : name,
        "report"   : report,
        "embedding": emb_np,
        "mask"     : binary_mask,
        "findings" : findings_data
    }

In [ ]:
all_results = []

for sample in matched[:2]:   
    result = run_full_pipeline(sample, model, ct_index)
    all_results.append(result)

print(f"\n{'='*55}")
print(f"DONE. Processed {len(all_results)} volumes.")
print(f"Output folders:")
for r in all_results:
    stem = r["name"].replace(".nii.gz", "")
    print(f"  results/{stem}/")
    print(f"    ├── report.txt")
    print(f"    ├── embedding.npz")
    print(f"    ├── localization_mask.nii.gz")
    print(f"    └── findings.json")

NameError: name 'matched' is not defined